# BITS Pilani - M.Tech AIML: Machine Learning Assignment 2
## Classification Models Training & Evaluation Pipeline

**Dataset**: Telco Customer Churn (Kaggle / IBM)
**Models**:
1. Logistic Regression
2. Decision Tree Classifier
3. K-Nearest Neighbor (KNN)
4. Naive Bayes (GaussianNB)
5. Support Vector Machine (SVM)

In [ ]:
import os
import urllib.request
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    classification_report, confusion_matrix
)

### 1. Load and Inspect Dataset

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(DATA_URL)
print(f'Raw shape: {df.shape}')
df.head()

### 2. Data Preprocessing & Cleaning

In [ ]:
# Clean TotalCharges numeric column
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(subset=['TotalCharges'], inplace=True)

# Drop non-predictive identifier
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])

target_col = 'Churn'
X = df.drop(columns=[target_col])
y = (df[target_col].str.strip() == 'Yes').astype(int)

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols = [c for c in X.columns if c not in num_cols]

print(f'Cleaned shape: {df.shape}')
print(f'Features ({len(X.columns)}): Numerical={len(num_cols)}, Categorical={len(cat_cols)}')

### 3. Stratified Train-Test Split (80/20)

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df[target_col])
test_df.to_csv('test_data.csv', index=False)
print(f'Train size: {len(train_df)}, Test size: {len(test_df)}')

X_train = train_df.drop(columns=[target_col])
y_train = (train_df[target_col].str.strip() == 'Yes').astype(int)
X_test = test_df.drop(columns=[target_col])
y_test = (test_df[target_col].str.strip() == 'Yes').astype(int)

### 4. Build Preprocessing Pipeline

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ],
    remainder='drop'
)
preprocessor.fit(X_train)
joblib.dump(preprocessor, 'model/preprocessor.joblib')
print('Preprocessor saved!')

### 5. Train and Evaluate All 5 Models

In [ ]:
models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, C=1.0, random_state=42), 'model/logistic_regression.joblib'),
    'Decision Tree': (DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=42), 'model/decision_tree.joblib'),
    'KNN': (KNeighborsClassifier(n_neighbors=15), 'model/knn.joblib'),
    'Naive Bayes': (GaussianNB(), 'model/naive_bayes.joblib'),
    'Support Vector Machine': (SVC(kernel='rbf', C=1.0, probability=True, cache_size=1000, random_state=42), 'model/svm.joblib')
}

results = []
for name, (clf, file_path) in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', clf)])
    pipe.fit(X_train, y_train)
    joblib.dump(pipe, file_path)
    
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    
    results.append({
        'ML Model Name': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'AUC': round(roc_auc_score(y_test, y_prob), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1': round(f1_score(y_test, y_pred), 4),
        'MCC': round(matthews_corrcoef(y_test, y_pred), 4)
    })

comparison_df = pd.DataFrame(results)
comparison_df